# Pensjonat Rybical Review Dashboard

This notebook reads reviews for the hotel, analyzes the good and bad points, and generates contextual, read-only responses based on previous human responses. Let's make sure this stays strictly **read-only**—no data goes back up to Google!


In [5]:
import json
import os
import re
import requests
from datetime import datetime

import pandas as pd
from IPython.display import display, HTML
from dotenv import load_dotenv
from google_auth_oauthlib.flow import InstalledAppFlow
from openai import OpenAI

load_dotenv()

# Google API Config
CLIENT_ID = os.getenv("GOOGLE_API_OAUTH_CLIENT_ID")
CLIENT_SECRET = os.getenv("GOOGLE_API_OAUTH_CLIENT_SECRET")
client_config = {
    "installed": {
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
    }
}

ACCOUNT_ID = "114674571764534564133"
LOCATION_ID = "8962787873732607458"
SCOPES = ["https://www.googleapis.com/auth/business.manage"]

# OpenAI Config
openai_client = OpenAI(api_key=os.getenv("CONFIG__OPENAI__KEY"))

# Pensjonat Rybical Context
HOTEL_CONTEXT = """Pensjonat Rybical is an intimate, family-run guesthouse and boutique retreat located directly on the shores of Lake Ryńskie in the village of Rybical, within Poland's beautiful Masurian Lake District. It is situated in a tranquil area near the towns of Ryn and Mikołajki.

The guesthouse is highly praised for its peaceful waterfront location and warm, homely atmosphere that offers a perfect escape from the city. Visitors consistently highlight the beautiful natural surroundings, the well-maintained property, and the hosts' dedication to making guests feel welcome.

Accommodations: Offers a variety of classic-style rooms, family studios, and cottages, many featuring lake views, private balconies or terraces, and soundproofing for extra comfort.

Water Activities: Features a private beach area, a private pier, and a small marina. Guests have access to kayaks, pedal boats, and can even rent motorboats right on the property.

Dining: The on-site restaurant serves a highly-rated breakfast with local products and fresh pastries. It specializes in traditional Polish cuisine and also offers vegetarian, vegan, and gluten-free options.

On-site Amenities: Includes a lakeside garden with plenty of sun loungers, an outdoor fireplace and barbecue area, a sauna for relaxation, and indoor entertainment like billiards and table tennis."""


In [6]:
def get_rybical_reviews():
    flow = InstalledAppFlow.from_client_config(client_config, SCOPES)
    creds = flow.run_local_server(port=0)
    headers = {
        "Authorization": f"Bearer {creds.token}",
        "Content-Type": "application/json",
    }
    url = f"https://mybusiness.googleapis.com/v4/accounts/{ACCOUNT_ID}/locations/{LOCATION_ID}/reviews"
    
    # Read-only operation!
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        reviews = data.get("reviews", [])
        print(f"Fetched {len(reviews)} reviews.")
        return reviews
        
    print(f"Error {response.status_code}: {response.text}")
    return []

my_reviews = get_rybical_reviews()

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=882398647440-shei634mkfbeujau8inuv4imeuv9n1dq.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A49857%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbusiness.manage&state=bxG2lmTgNI2LcqGlv3HLtfSimYQ7XD&code_challenge=MyxSqViYAM35KuF-3cvbMNvIOtBuj0gXB-TLoW5NKUc&code_challenge_method=S256&access_type=offline
Fetched 50 reviews.


In [7]:
def analyze_review_and_suggest_response(review_text: str, rating: str, reviewer: str, examples: list) -> dict:
    if not review_text or str(review_text).strip() == "":
        return {
            "good_points": "", 
            "bad_points": "", 
            "suggested_response": "Dziękujemy za pozytywną ocenę! Zapraszamy ponownie." if rating in ["FOUR", "FIVE"] else "Dziękujemy za opinię."
        }
        
    # Extract Original Text from Google Translation block if it exists
    # Google format is usually: "(Translated by Google) ... \n\n(Original)\n..."
    original_text_match = re.search(r'\(Original\)\s*(.*)', review_text, re.DOTALL | re.IGNORECASE)
    if original_text_match:
        review_text = original_text_match.group(1).strip()
    else:
        # Sometimes there's just "(Translated by Google) ... " but no original
        review_text = re.sub(r'\(Translated by Google\)\s*', '', review_text).strip()

    examples_text = "\n\n".join([f"Opinia Gościa: {ex['comment']}\nTwoja Odpowiedź: {ex['our_response']}" for ex in examples])

    prompt = f"""Jesteś właścicielem Pensjonatu Rybical. Analizujesz opinię gościa i przygotowujesz szkic odpowiedzi.
    
Oto kontekst Twojego pensjonatu:
{HOTEL_CONTEXT}

Oto przykłady Twoich wcześniejszych odpowiedzi (do naśladowania stylu):
{examples_text}

Zadanie:
1. Wypunktuj krótko co gość ocenił jako PLUSY i MINUSY w opinii (krótkie równoważniki zdań po polsku).
2. Przygotuj sugerowaną odpowiedź na opinię w języku polskim.
   - Odpowiedź musi być uprzejma, profesjonalna i ciepła.
   - Musi pasować stylem i tonem do Twoich wcześniejszych odpowiedzi.
   - Odnieś się uprzejmie do ewentualnych uwag (minusów).
   - Podziękuj za pochwały (plusy).
   - Zwróć się do recenzenta po imieniu, jeśli pasuje ({reviewer}).
   
Odpowiedz TYLKO i WYŁĄCZNIE obiektem JSON w tym dokładnym formacie (żadnego formatowania markdown, żadnych bloków kodu):
{{
  "good_points": "...",
  "bad_points": "...",
  "suggested_response": "..."
}}

Tekst opinii (wersja oryginalna):
{review_text}
Ocena (Rating): {rating}
"""
    
    try:
        resp = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
        )
        text = resp.choices[0].message.content.strip()
        if text.startswith("```"):
            text = re.sub(r"^```(?:json)?\s*|```$", "", text, flags=re.MULTILINE).strip()
            
        return json.loads(text)
    except Exception as e:
        return {"good_points": "Analysis Error", "bad_points": "Analysis Error", "suggested_response": f"Generation Error: {str(e)}"}

def process_unanswered_reviews(reviews):
    # 1. Extract examples for few-shot prompting from answered reviews
    examples = []
    for r in reviews:
        comment = r.get("comment", "")
        reply = r.get("reviewReply", {}).get("comment", "")
        if comment and reply and len(examples) < 10:
            examples.append({"comment": comment, "our_response": reply})

    # 2. Process unanswered reviews
    processed = []
    for r in reviews:
        comment = r.get("comment", "")
        create_time = r.get("createTime", "")
        try:
            dt = datetime.fromisoformat(create_time.replace("Z", "+00:00"))
            date_str = dt.strftime("%Y-%m-%d")
        except Exception:
            date_str = create_time
            
        rating = r.get("starRating", "")
        reviewer = (r.get("reviewer") or {}).get("displayName", "")
        
        reply_obj = r.get("reviewReply") or {}
        our_response = reply_obj.get("comment", "") if reply_obj else ""
        
        # Only process reviews that we haven't responded to yet
        if not our_response:
            print(f"Generating response for {reviewer}...")
            analysis = analyze_review_and_suggest_response(comment, rating, reviewer, examples)
            
            processed.append({
                "date": date_str,
                "reviewer": reviewer,
                "rating": rating,
                "comment": comment,
                "good_points": analysis.get("good_points", ""),
                "bad_points": analysis.get("bad_points", ""),
                "suggested_response": analysis.get("suggested_response", "")
            })
            
    return processed

unanswered_reviews = process_unanswered_reviews(my_reviews)

Generating response for Piotr Ostapczuk...
Generating response for Grzegorz Sujkowski...
Generating response for Aneta Biela...
Generating response for Golden Eye...
Generating response for Marek T...
Generating response for Bo Wi...
Generating response for Nicole Ehlers...
Generating response for Elke St....
Generating response for Aleksandra Mozolewska...
Generating response for Elżbieta Janczyk-Strzała...
Generating response for R...
Generating response for Sabina Pająk-Danicka...
Generating response for Patrick Palm...
Generating response for Krzysztof Tkaczyk...
Generating response for Maciej Kluz...
Generating response for Ben Woods...
Generating response for Marek Olewicz...
Generating response for Monika Bernatowicz...
Generating response for Kinga Osińska...


In [8]:
def render_dashboard(reviews_data):
    if not reviews_data:
        display(HTML("<h3 style='color: green;'>✅ Brak nowych opinii do obsłużenia! (No new reviews to handle!)</h3>"))
        return

    html = "<div style='font-family: Arial, sans-serif; max-width: 900px; margin: 0 auto;'>"
    html += "<h2 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;'>Pulpit Opinii Pensjonat Rybical (Review Dashboard)</h2>"
    html += f"<p>Znaleziono {len(reviews_data)} opinii bez odpowiedzi.</p>"
    
    for r in reviews_data:
        stars = "⭐" * {"ONE": 1, "TWO": 2, "THREE": 3, "FOUR": 4, "FIVE": 5}.get(r['rating'], 0)
        
        # Wyciągamy oryginalny tekst do podglądu UI
        comment_display = r['comment']
        original_text_match = re.search(r'\(Original\)\s*(.*)', comment_display, re.DOTALL | re.IGNORECASE)
        if original_text_match:
            comment_display = original_text_match.group(1).strip()
        else:
            comment_display = re.sub(r'\(Translated by Google\)\s*', '', comment_display).strip()

        if not comment_display:
            comment_display = 'Brak opisu'

        # Format HTML template
        html += f"""
        <div style='border: 1px solid #ddd; border-radius: 8px; padding: 20px; margin-bottom: 24px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); background-color: white;'>
            <div style='display: flex; justify-content: space-between; margin-bottom: 12px; align-items: center;'>
                <strong style='font-size: 1.1em;'>{r['reviewer']}</strong>
                <span style='color: #7f8c8d; font-size: 0.9em;'>{r['date']}</span>
            </div>
            
            <div style='margin-bottom: 12px; font-size: 1.1em;'>
                {stars}
            </div>
            
            <div style='background-color: #f8f9fa; padding: 12px; border-left: 4px solid #ced4da; margin-bottom: 20px; font-style: italic; color: #495057;'>
                "{comment_display}"
            </div>
            
            <div style='display: flex; gap: 20px; margin-bottom: 20px;'>
                <div style='flex: 1; background-color: #e8f5e9; padding: 12px; border-radius: 6px; border-left: 4px solid #4caf50;'>
                    <h4 style='margin: 0 0 8px 0; color: #2e7d32; font-size: 0.95em;'>✅ Plusy</h4>
                    <p style='margin: 0; font-size: 0.9em; line-height: 1.4;'>{r['good_points'] if r['good_points'] else 'Brak'}</p>
                </div>
                <div style='flex: 1; background-color: #ffebee; padding: 12px; border-radius: 6px; border-left: 4px solid #f44336;'>
                    <h4 style='margin: 0 0 8px 0; color: #c62828; font-size: 0.95em;'>❌ Minusy</h4>
                    <p style='margin: 0; font-size: 0.9em; line-height: 1.4;'>{r['bad_points'] if r['bad_points'] else 'Brak'}</p>
                </div>
            </div>
            
            <div style='background-color: #e3f2fd; padding: 16px; border-radius: 6px; border-left: 4px solid #2196f3;'>
                <h4 style='margin: 0 0 10px 0; color: #1565c0; display: flex; align-items: center; font-size: 1em;'>
                    <span style='margin-right: 8px;'>🤖</span> Sugerowana Odpowiedź
                </h4>
                <p style='margin: 0; white-space: pre-wrap; line-height: 1.5; color: #000;'>{r['suggested_response']}</p>
            </div>
        </div>
        """
        
    html += "</div>"
    display(HTML(html))

render_dashboard(unanswered_reviews)
